# Test 6.2v2: K-Sparse Ablation + Fair Baseline Comparison

**Гипотеза:** V4 K-Sparse Chaos превосходит Dense ReLU при одинаковой capacity (latent_dim=128).

**План:**
- K-Sparse Ablation: K ∈ {4, 8, 16, 32, 64, 96, 112}, latent_dim=128
- Fair Baseline Comparison: Dense_ReLU_64 vs Dense_ReLU_128 vs V4_KSparse_128, N=10
- Legacy Multiple Runs: Dense_ReLU vs KSparse_Chaos, N=5
- Henon Map Generalization: проверка на второй хаотической системе
- Dead Neuron Tracking: мониторинг нейронов в процессе обучения

In [13]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pandas as pd
import json
from datetime import datetime
import os

print(f"TF version: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

TF version: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [14]:
def logistic_map(x, r=3.99):
    return r * x * (1 - x)

def generate_logistic_map_image(image_size=28, initial_value=0.4, r=3.99):
    iterations = image_size * image_size
    x = initial_value
    seq = []
    for _ in range(iterations):
        x = logistic_map(x, r)
        seq.append(x)
    img = np.array(seq).reshape((image_size, image_size))
    return img

def generate_logistic_dataset(num_images, image_size=28, r=3.99, fixed_initial=False):
    dataset = []
    for _ in range(num_images):
        init_val = 0.4 if fixed_initial else np.random.rand()
        img = generate_logistic_map_image(image_size=image_size, initial_value=init_val, r=r)
        dataset.append(img)
    return np.array(dataset)[..., np.newaxis].astype('float32')

def henon_map(x, y, a=1.4, b=0.3):
    x = np.clip(x, -2.0, 2.0)
    y = np.clip(y, -2.0, 2.0)
    x_new = 1 - a * x**2 + y
    y_new = b * x
    return x_new, y_new

def generate_henon_image(image_size=28, x0=0.1, y0=0.1, a=1.4, b=0.3):
    iterations = image_size * image_size
    x, y = x0, y0
    points = []
    for _ in range(iterations):
        x, y = henon_map(x, y, a, b)
        normalized = (x + 2.0) / 4.0
        points.append(np.clip(normalized, 0, 1))
    img = np.array(points).reshape((image_size, image_size))
    return img

def generate_henon_dataset(num_images, image_size=28, a=1.4, b=0.3):
    dataset = []
    for _ in range(num_images):
        x0 = np.random.uniform(-0.5, 0.5)
        y0 = np.random.uniform(-0.5, 0.5)
        img = generate_henon_image(image_size, x0, y0, a, b)
        dataset.append(img)
    return np.array(dataset)[..., np.newaxis].astype('float32')

In [15]:
class KSparseLayer(layers.Layer):
    def __init__(self, k=32, **kwargs):
        super(KSparseLayer, self).__init__(**kwargs)
        self.k = k

    def call(self, inputs, training=None):
        batch_size = tf.shape(inputs)[0]
        latent_dim = tf.shape(inputs)[1]
        values, indices = tf.nn.top_k(tf.abs(inputs), k=self.k, sorted=False)
        mask = tf.reduce_sum(
            tf.one_hot(indices, latent_dim, dtype=inputs.dtype),
            axis=1
        )
        return inputs * mask

    def get_config(self):
        config = super().get_config()
        config.update({"k": self.k})
        return config


class TargetVarianceRegularizer(layers.Layer):
    def __init__(self, lambda_reg=0.01, target_variance=0.1, **kwargs):
        super(TargetVarianceRegularizer, self).__init__(**kwargs)
        self.lambda_reg = lambda_reg
        self.target_variance = target_variance

    def call(self, inputs):
        current_variance = tf.math.reduce_variance(inputs, axis=0)
        mean_variance = tf.reduce_mean(current_variance)
        variance_penalty = self.lambda_reg * tf.square(
            mean_variance - self.target_variance
        )
        self.add_loss(variance_penalty)
        return inputs

    def get_config(self):
        config = super().get_config()
        config.update({
            "lambda_reg": self.lambda_reg,
            "target_variance": self.target_variance
        })
        return config


def chaos_activation(x):
    return tf.sin(8.0 * x) + 0.5 * tf.tanh(4.0 * x)


def build_ksparse_chaos_ae(image_size=(28, 28), latent_dim=128, k_active=32):
    input_img = keras.Input(shape=(*image_size, 1))
    x = layers.Flatten()(input_img)

    x = layers.Dense(256)(x)
    x = layers.Activation(chaos_activation)(x)
    x = layers.Dropout(0.2)(x)

    latent_pre = layers.Dense(latent_dim, name='latent_pre')(x)
    latent_pre = layers.Activation(chaos_activation)(latent_pre)

    latent = KSparseLayer(k=k_active, name='latent_ksparse')(latent_pre)
    latent = TargetVarianceRegularizer(
        lambda_reg=0.01,
        target_variance=0.1
    )(latent)

    encoder = keras.Model(input_img, latent, name='ksparse_chaos_encoder')

    x = layers.Dense(256)(latent)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(chaos_activation)(x)
    x = layers.Dropout(0.1)(x)

    decoded = layers.Dense(np.prod(image_size), activation='sigmoid')(x)
    decoded = layers.Reshape((*image_size, 1))(decoded)

    autoencoder = keras.Model(input_img, decoded, name='ksparse_chaos_autoencoder')
    autoencoder.compile(optimizer='adam', loss='mse')

    return autoencoder, encoder


def build_dense_relu_ae(image_size=(28, 28), latent_dim=64):
    h, w = image_size
    input_img = keras.Input(shape=(h, w, 1))
    x = layers.Flatten()(input_img)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    latent = layers.Dense(latent_dim, activation="relu", name="latent")(x)

    encoder = keras.Model(input_img, latent, name="dense_encoder")

    x = layers.Dense(128, activation="relu")(latent)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(h * w, activation="sigmoid")(x)
    decoded = layers.Reshape((h, w, 1))(x)

    autoencoder = keras.Model(input_img, decoded)
    autoencoder.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')

    return autoencoder, encoder

In [16]:
def analyze_latent_statistics(encoder, images, zero_threshold=1e-6):
    latents = encoder.predict(images, verbose=0)

    variance_per_dim = np.var(latents, axis=0)
    mean_variance = float(np.mean(variance_per_dim))

    dead_mask = np.all(np.abs(latents) < zero_threshold, axis=0)
    dead_neurons = int(np.sum(dead_mask))
    total_neurons = latents.shape[1]

    zero_mask = np.abs(latents) < zero_threshold
    overall_sparsity = np.mean(zero_mask)

    active_per_sample = np.sum(~zero_mask, axis=1)
    mean_active = np.mean(active_per_sample)

    variance_active = []
    for dim in range(latents.shape[1]):
        dim_values = latents[:, dim]
        active_mask = ~zero_mask[:, dim]
        if np.sum(active_mask) > 1:
            active_values = dim_values[active_mask]
            variance_active.append(np.var(active_values))

    mean_variance_active = np.mean(variance_active) if variance_active else 0.0

    return {
        'variance_per_dim': variance_per_dim,
        'mean_variance': mean_variance,
        'dead_neurons': dead_neurons,
        'total_neurons': total_neurons,
        'dead_percentage': dead_neurons / total_neurons,
        'overall_sparsity': overall_sparsity,
        'mean_active_neurons': mean_active,
        'mean_variance_active': mean_variance_active,
        'latents': latents
    }


def track_dead_neurons_over_time(model, encoder, images, epochs=50, batch_size=64):
    trajectory = []
    for epoch in range(epochs):
        model.fit(images, images, epochs=1, batch_size=batch_size, verbose=0)
        s = analyze_latent_statistics(encoder, images[:200])
        trajectory.append({
            'epoch': epoch,
            'dead_neurons': s['dead_neurons'],
            'mean_variance': s['mean_variance'],
            'sparsity': s['overall_sparsity']
        })
    return trajectory

In [17]:
train_images = generate_logistic_dataset(2000, fixed_initial=False)
test_images = generate_logistic_dataset(500, fixed_initial=False)

k_values = [4, 8, 16, 32, 64, 96, 112]
latent_dim = 128
k_results = {}

print(f"Testing K values: {k_values}")
print(f"Latent dimension: {latent_dim}")

for k in k_values:
    print(f"\n[K={k}] Training...")
    ae, enc = build_ksparse_chaos_ae(latent_dim=latent_dim, k_active=k)

    history = ae.fit(
        train_images, train_images,
        epochs=10,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )

    s = analyze_latent_statistics(enc, test_images)

    k_results[k] = {
        'stats': s,
        'val_loss': history.history['val_loss'][-1],
        'sparsity': (latent_dim - k) / latent_dim
    }

    print(f"  Variance: {s['mean_variance']:.6f}")
    print(f"  Dead neurons: {s['dead_neurons']}/{latent_dim}")
    print(f"  Sparsity: {k_results[k]['sparsity']:.1%}")

# Results table
header = f"\n{'K':<5} {'Sparsity':<10} {'Variance':<12} {'Dead':<10} {'Val Loss'}"
print(header)
print("-" * 50)

for k in k_values:
    r = k_results[k]
    print(f"{k:<5} {r['sparsity']:<10.1%} {r['stats']['mean_variance']:<12.4f} "
          f"{r['stats']['dead_neurons']}/{latent_dim:<6} {r['val_loss']:.4f}")

variances = [k_results[k]['stats']['mean_variance'] for k in k_values]
violations = sum(1 for i in range(len(variances)-1) if variances[i+1] < variances[i])
print(f"\nTrend violations: {violations}/{len(variances)-1}")

Testing K values: [4, 8, 16, 32, 64, 96, 112]
Latent dimension: 128

[K=4] Training...
Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - loss: 0.1367 - val_loss: 0.1344
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.1331 - val_loss: 0.1307
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.1302 - val_loss: 0.1274
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.1274 - val_loss: 0.1250
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.1254 - val_loss: 0.1232
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.1236 - val_loss: 0.1219
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.1221 - val_loss: 0.1206
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.1210 - val_loss: 0.1198
Epoch 9/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1201 - val_loss: 0.1190
Epoch 10/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1194 - val_loss: 0.1186
  Variance: 0.067974
  Dead neurons: 0/128
  Sparsity: 96.9%



In [18]:
num_runs = 10
architectures = {
    'Dense_ReLU_64': lambda: build_dense_relu_ae(latent_dim=64),
    'Dense_ReLU_128': lambda: build_dense_relu_ae(latent_dim=128),
    'V4_KSparse_128': lambda: build_ksparse_chaos_ae(latent_dim=128, k_active=32)
}

fair_results = {name: [] for name in architectures.keys()}

for arch_name, builder in architectures.items():
    print(f"\nArchitecture: {arch_name}")

    for run in range(num_runs):
        np.random.seed(run)
        tf.random.set_seed(run)

        ae, enc = builder()
        history = ae.fit(
            train_images, train_images,
            epochs=10,
            batch_size=64,
            validation_split=0.1,
            verbose=0
        )

        s = analyze_latent_statistics(enc, test_images)

        fair_results[arch_name].append({
            'run': run,
            'variance': s['mean_variance'],
            'dead_neurons': s['dead_neurons'],
            'total_neurons': s['total_neurons'],
            'dead_percentage': s['dead_percentage'],
            'val_loss': history.history['val_loss'][-1]
        })

        print(f"  Run {run+1}/{num_runs}: var={s['mean_variance']:.4f}, "
              f"dead={s['dead_neurons']}/{s['total_neurons']} "
              f"({s['dead_percentage']:.0%}), "
              f"loss={history.history['val_loss'][-1]:.4f}")


Architecture: Dense_ReLU_64
  Run 1/10: var=0.3309, dead=0/64 (0%), loss=0.1186
  Run 2/10: var=0.3256, dead=0/64 (0%), loss=0.1178
  Run 3/10: var=0.3166, dead=0/64 (0%), loss=0.1180
  Run 4/10: var=0.3362, dead=0/64 (0%), loss=0.1175
  Run 5/10: var=0.3228, dead=0/64 (0%), loss=0.1181
  Run 6/10: var=0.3353, dead=0/64 (0%), loss=0.1186
  Run 7/10: var=0.3341, dead=0/64 (0%), loss=0.1178
  Run 8/10: var=0.3342, dead=0/64 (0%), loss=0.1185
  Run 9/10: var=0.3424, dead=0/64 (0%), loss=0.1184
  Run 10/10: var=0.3335, dead=0/64 (0%), loss=0.1181

Architecture: Dense_ReLU_128
  Run 1/10: var=0.2428, dead=0/128 (0%), loss=0.1185
  Run 2/10: var=0.2536, dead=0/128 (0%), loss=0.1216
  Run 3/10: var=0.2457, dead=0/128 (0%), loss=0.1205
  Run 4/10: var=0.2537, dead=0/128 (0%), loss=0.1211
  Run 5/10: var=0.2455, dead=0/128 (0%), loss=0.1207
  Run 6/10: var=0.2612, dead=0/128 (0%), loss=0.1215
  Run 7/10: var=0.2360, dead=0/128 (0%), loss=0.1191
  Run 8/10: var=0.2434, dead=0/128 (0%), loss=0.1

In [19]:
summary = {}
for arch_name, runs in fair_results.items():
    variances = [r['variance'] for r in runs]
    dead_counts = [r['dead_neurons'] for r in runs]
    dead_pcts = [r['dead_percentage'] for r in runs]
    losses = [r['val_loss'] for r in runs]

    var_mean = np.mean(variances)
    var_std = np.std(variances)
    var_ci = 1.96 * var_std / np.sqrt(len(variances))

    print(f"\n{arch_name}:")
    print(f"  Variance:     {var_mean:.6f} +/- {var_std:.6f} (CI: +/-{var_ci:.6f})")
    print(f"  Dead neurons: {np.mean(dead_counts):.1f} +/- {np.std(dead_counts):.1f} ({np.mean(dead_pcts):.1%})")
    print(f"  Val loss:     {np.mean(losses):.6f} +/- {np.std(losses):.6f}")

    summary[arch_name] = {
        'variances': variances,
        'dead_pcts': dead_pcts,
        'var_mean': var_mean,
        'var_std': var_std,
        'var_ci': var_ci,
        'dead_pct_mean': np.mean(dead_pcts),
        'loss_mean': np.mean(losses)
    }

# Dense_64 vs Dense_128
print("\n--- Dense_64 vs Dense_128 (dimension effect) ---")
vars_64 = summary['Dense_ReLU_64']['variances']
vars_128 = summary['Dense_ReLU_128']['variances']
t_stat, p_value = stats.ttest_ind(vars_64, vars_128)
improvement = np.mean(vars_128) / np.mean(vars_64)
print(f"  Variance: {np.mean(vars_64):.4f} -> {np.mean(vars_128):.4f} ({improvement:.2f}x)")
print(f"  t={t_stat:.4f}, p={p_value:.6f}")

# Dense_128 vs V4_128 (FAIR)
print("\n--- Dense_128 vs V4_128 (FAIR comparison, same capacity) ---")
vars_dense = summary['Dense_ReLU_128']['variances']
vars_v4 = summary['V4_KSparse_128']['variances']
dead_dense = summary['Dense_ReLU_128']['dead_pct_mean']
dead_v4 = summary['V4_KSparse_128']['dead_pct_mean']

t_stat, p_value = stats.ttest_ind(vars_dense, vars_v4)
improvement = np.mean(vars_v4) / np.mean(vars_dense)
print(f"  Variance: {np.mean(vars_dense):.4f} -> {np.mean(vars_v4):.4f} ({improvement:.2f}x)")
print(f"  Dead neurons: {dead_dense:.1%} -> {dead_v4:.1%}")
print(f"  t={t_stat:.4f}, p={p_value:.6f}")
assert p_value < 0.05, "V4 should significantly outperform Dense_128"
assert improvement > 1.5, "V4 should have at least 1.5x higher variance"


Dense_ReLU_64:
  Variance:     0.331161 +/- 0.007115 (CI: +/-0.004410)
  Dead neurons: 0.0 +/- 0.0 (0.0%)
  Val loss:     0.118144 +/- 0.000377

Dense_ReLU_128:
  Variance:     0.247460 +/- 0.007000 (CI: +/-0.004339)
  Dead neurons: 0.0 +/- 0.0 (0.0%)
  Val loss:     0.120209 +/- 0.001115

V4_KSparse_128:
  Variance:     0.417833 +/- 0.003002 (CI: +/-0.001860)
  Dead neurons: 0.0 +/- 0.0 (0.0%)
  Val loss:     0.119452 +/- 0.000099

--- Dense_64 vs Dense_128 (dimension effect) ---
  Variance: 0.3312 -> 0.2475 (0.75x)
  t=25.1572, p=0.000000

--- Dense_128 vs V4_128 (FAIR comparison, same capacity) ---
  Variance: 0.2475 -> 0.4178 (1.69x)
  Dead neurons: 0.0% -> 0.0%
  t=-67.1038, p=0.000000


In [20]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

arch_names = list(fair_results.keys())
x = np.arange(len(arch_names))

colors = {
    'Dense_ReLU_64': 'lightcoral',
    'Dense_ReLU_128': 'skyblue',
    'V4_KSparse_128': 'lightgreen'
}
color_list = [colors[name] for name in arch_names]

variances_mean = []
variances_std = []
dead_pct_mean = []
dead_pct_std = []
loss_mean = []
loss_std = []

for arch in arch_names:
    vars_list = [r['variance'] for r in fair_results[arch]]
    dead_list = [r['dead_percentage'] * 100 for r in fair_results[arch]]
    loss_list = [r['val_loss'] for r in fair_results[arch]]

    variances_mean.append(np.mean(vars_list))
    variances_std.append(np.std(vars_list))
    dead_pct_mean.append(np.mean(dead_list))
    dead_pct_std.append(np.std(dead_list))
    loss_mean.append(np.mean(loss_list))
    loss_std.append(np.std(loss_list))

# Variance
axes[0, 0].bar(x, variances_mean, yerr=variances_std, capsize=5,
               alpha=0.7, color=color_list, edgecolor='black', linewidth=1.5)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(arch_names, rotation=15, ha='right', fontsize=10)
axes[0, 0].set_ylabel('Mean Variance', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Variance Comparison\n(FAIR: Same capacity for 128-dim models)',
                      fontweight='bold', fontsize=13)
axes[0, 0].grid(True, alpha=0.3, axis='y')

if len(arch_names) >= 3:
    axes[0, 0].annotate('', xy=(2, variances_mean[2]), xytext=(1, variances_mean[1]),
                         arrowprops=dict(arrowstyle='<->', color='red', lw=2.5))
    imp = variances_mean[2] / variances_mean[1]
    mid_y = (variances_mean[1] + variances_mean[2]) / 2
    axes[0, 0].text(1.5, mid_y, f'{imp:.2f}x\nFAIR',
                    ha='center', va='center', fontsize=12, color='red', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow',
                              edgecolor='red', linewidth=2, alpha=0.8))

# Dead Neurons
axes[0, 1].bar(x, dead_pct_mean, yerr=dead_pct_std, capsize=5,
               alpha=0.7, color=color_list, edgecolor='black', linewidth=1.5)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(arch_names, rotation=15, ha='right', fontsize=10)
axes[0, 1].set_ylabel('Dead Neurons (%)', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Dead Neurons Comparison', fontweight='bold', fontsize=13)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Loss
axes[1, 0].bar(x, loss_mean, yerr=loss_std, capsize=5,
               alpha=0.7, color=color_list, edgecolor='black', linewidth=1.5)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(arch_names, rotation=15, ha='right', fontsize=10)
axes[1, 0].set_ylabel('Validation Loss (MSE)', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Reconstruction Quality', fontweight='bold', fontsize=13)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Summary table
axes[1, 1].axis('off')
table_data = []
for idx, arch in enumerate(arch_names):
    ld = 64 if '64' in arch else 128
    table_data.append([
        arch.replace('_', '\n'),
        f"{ld}",
        f"{variances_mean[idx]:.3f}",
        f"{dead_pct_mean[idx]:.1f}%"
    ])

table = axes[1, 1].table(cellText=table_data,
                          colLabels=['Architecture', 'Dims', 'Variance', 'Dead %'],
                          cellLoc='center', loc='center', bbox=[0, 0.3, 1, 0.6])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)
for i, arch in enumerate(arch_names):
    if '128' in arch:
        for j in range(4):
            table[(i+1, j)].set_facecolor('lightyellow')
            table[(i+1, j)].set_edgecolor('orange')
            table[(i+1, j)].set_linewidth(2)

axes[1, 1].set_title('Summary Table', fontweight='bold', fontsize=13)

plt.suptitle(f'Fair Baseline Comparison (N={num_runs} runs)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('fair_baseline_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

In [21]:
logistic_train = generate_logistic_dataset(2000, fixed_initial=False)
henon_train = generate_henon_dataset(2000)
logistic_test = generate_logistic_dataset(500, fixed_initial=False)
henon_test = generate_henon_dataset(500)

henon_results = {}

for dataset_name, t_data, te_data in [
    ('Logistic', logistic_train, logistic_test),
    ('Henon', henon_train, henon_test)
]:
    print(f"\nTraining on {dataset_name} Map...")
    ae, enc = build_ksparse_chaos_ae(latent_dim=128, k_active=32)
    history = ae.fit(
        t_data, t_data,
        epochs=10,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )
    s = analyze_latent_statistics(enc, te_data)
    henon_results[dataset_name] = {
        'encoder': enc,
        'stats': s,
        'val_loss': history.history['val_loss'][-1]
    }
    print(f"  Variance: {s['mean_variance']:.6f}")
    print(f"  Dead neurons: {s['dead_neurons']}/{s['total_neurons']}")

log_var = henon_results['Logistic']['stats']['mean_variance']
hen_var = henon_results['Henon']['stats']['mean_variance']
variance_ratio = max(log_var, hen_var) / min(log_var, hen_var)
print(f"\nVariance ratio (Henon/Logistic): {hen_var/log_var:.2f}x")
assert henon_results['Logistic']['stats']['dead_neurons'] == 0
assert henon_results['Henon']['stats']['dead_neurons'] == 0
assert variance_ratio < 3.0, f"Variance too different: {variance_ratio:.2f}x"


Training on Logistic Map...
Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - loss: 0.1378 - val_loss: 0.1337
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.1342 - val_loss: 0.1306
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1314 - val_loss: 0.1281
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1288 - val_loss: 0.1262
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1266 - val_loss: 0.1243
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1249 - val_loss: 0.1230
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1234 - val_loss: 0.1218
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1222 - val_loss: 0.1210
Epoch 9/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1214 - val_loss: 0.1207
Epoch 10/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1206 - val_loss: 0.1200
  Variance: 0.417850
  Dead neurons: 0/128

Training on Henon Map...
Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step 

In [22]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, (name, te_data) in enumerate([
    ('Logistic', logistic_test),
    ('Henon', henon_test)
]):
    enc = henon_results[name]['encoder']
    latents = enc.predict(te_data[:100], verbose=0)

    axes[0, idx].scatter(latents[:, 0], latents[:, 1],
                       alpha=0.6, s=30, edgecolors='black', linewidths=0.5)
    axes[0, idx].set_xlabel('Dim 0')
    axes[0, idx].set_ylabel('Dim 1')
    axes[0, idx].set_title(f'{name} Map - Latent Space', fontweight='bold')
    axes[0, idx].grid(True, alpha=0.3)

    variance = np.var(latents, axis=0)
    axes[1, idx].hist(np.log10(variance + 1e-10), bins=20, alpha=0.7,
                    edgecolor='black')
    axes[1, idx].set_xlabel('Log10(Variance)')
    axes[1, idx].set_ylabel('Frequency')
    axes[1, idx].set_title(f'{name} Map - Variance Distribution', fontweight='bold')
    axes[1, idx].axvline(x=np.log10(henon_results[name]['stats']['mean_variance']),
                        color='red', linestyle='--', linewidth=2, label='Mean')
    axes[1, idx].legend()
    axes[1, idx].grid(True, alpha=0.3)

plt.suptitle('Generalization Test: Logistic vs Henon Map',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('henon_generalization.png', dpi=300, bbox_inches='tight')
plt.close()

In [23]:
images = generate_logistic_dataset(1000, fixed_initial=False)
ae, enc = build_ksparse_chaos_ae(latent_dim=128, k_active=32)
trajectory = track_dead_neurons_over_time(ae, enc, images, epochs=30)

header = f"{'Epoch':<8} {'Dead':<8} {'Variance':<12} {'Sparsity'}"
print(header)
print('-' * 40)
for t in trajectory[::5]:
    print(f"{t['epoch']:<8} {t['dead_neurons']:<8} {t['mean_variance']:<12.6f} {t['sparsity']:.3f}")

final_dead = trajectory[-1]['dead_neurons']
assert final_dead == 0, "Should maintain 0 dead neurons"

final_variance = trajectory[-1]['mean_variance']
initial_variance = trajectory[5]['mean_variance']
assert final_variance > initial_variance * 0.8, "Variance should not collapse"

Epoch    Dead     Variance     Sparsity
----------------------------------------
0        0        0.427493     0.750
5        0        0.419953     0.750
10       0        0.415423     0.750
15       0        0.421607     0.750
20       0        0.418969     0.750
25       0        0.416780     0.750


In [24]:
epochs_list = [t['epoch'] for t in trajectory]
dead_list = [t['dead_neurons'] for t in trajectory]
variance_list = [t['mean_variance'] for t in trajectory]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(epochs_list, dead_list, 'o-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Dead Neurons')
axes[0].set_title('Dead Neurons Over Time', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_list, variance_list, 'o-', linewidth=2, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Variance')
axes[1].set_title('Variance Over Time', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('K-Sparse Chaos AE: Training Stability',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_stability.png', dpi=300, bbox_inches='tight')
plt.close()